In [1]:
from typing import Union
import numpy as np
from numpy.random import MT19937
from numpy.random import RandomState, SeedSequence
from scipy.stats import moment, bootstrap, norm
import json

In [2]:
# Mersenne twister init
SEED = 6
GENERATOR = RandomState(MT19937(SeedSequence(SEED)))

# confidence level
BETA = 0.95

In [12]:
# sample generation

from random import sample


def p(
    x: Union[float, np.ndarray],
    theta: float
) -> Union[float, np.ndarray]:
    return (theta - 1) / x ** theta * (x >= 1)

def F(
    x: Union[float, np.ndarray],
    theta: float
) -> Union[float, np.ndarray]:
    return np.pow(1 - x, (-theta + 1))

def F_reversed(
    x: np.ndarray,
    theta: float
) -> np.ndarray:
    return np.pow(1 - x, -1 / (theta + 1))

x = GENERATOR.uniform(size=100)
sample = F_reversed(x, theta=6)

In [13]:
intervals = dict()

In [14]:
# asymp conf interval 
intervals["asymptotic_theta"] = (
    1 + np.log(sample).sum() / 10 - norm.ppf((1 + BETA) / 2),
    1 + np.log(sample).sum() / 10 - norm.ppf((1 - BETA) / 2)
)

In [15]:
# asymp conf interval 
intervals["asymptotic_median"] = (
    2 ** (1 / np.log(sample).sum() / 10 - norm.ppf((1 + BETA) / 2)),
    2 ** (1 / np.log(sample).sum() / 10 - norm.ppf((1 - BETA) / 2))
)

In [ ]:
# non-parametric bootsrap confidence interval
result = bootstrap(
    sample.reshape(1, -1),
    lambda x : 1 + 100 / np.log(x).sum(),
    n_resamples=10000,
    confidence_level=BETA,
    method='basic',
    rng=GENERATOR
)

intervals["bootstrap"] = (
    result.confidence_interval.low,
    result.confidence_interval.high
)

In [ ]:
# parametric bootstrap

theta_hat = 1 + 100 / np.log(sample).sum()

boot_samples = F_reversed(GENERATOR.uniform(size=(10000, 100)), theta=6)

boot_thetas = 1 + 100 / np.log(boot_samples).sum(axis=1)

deltas = boot_thetas - theta_hat
delta_bounds = np.quantile(deltas, [(1 + BETA) / 2, (1 - BETA) / 2])
ci_basic = theta_hat - delta_bounds

intervals["bootstrap_param"] = (ci_basic[1])

Параметрический (пивотный):      [5.1337, 7.9247]
